# Problem 5 (100 points)

Batch normalization is a technique that normalizes each feature across a mini-batch, then applies a learnable affine transformation. It accelerates training, reduces sensitivity to initialization, and acts as a mild regularizer. In this problem, you will implement batch normalization’s forward pass, backward pass, and running-statistics tracking **from scratch**, without using `nn.BatchNorm1d` or `F.batch_norm`.

We use the following notation in this problem.
- $B$ — batch size.
- $D$ — number of features.
- $x \in \mathbb{R}^{B \times D}$ — input activations.
- $\mu_B = \frac{1}{B}\sum_{i=1}^{B} x_i \in \mathbb{R}^D$ — batch mean (per feature).
- $\sigma_B^2 = \frac{1}{B}\sum_{i=1}^{B}(x_i - \mu_B)^2 \in \mathbb{R}^D$ — batch variance (per feature, **biased** estimate).
- $\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ — normalized activations.
- $y_i = \gamma \hat{x}_i + \beta$ — output, where $\gamma, \beta \in \mathbb{R}^D$ are learnable.
- $\epsilon = 10^{-5}$ — small constant for numerical stability.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**
- Do **NOT** use `nn.BatchNorm1d`, `nn.BatchNorm2d`, or `F.batch_norm` in any part of this problem.

## Part 1 (5 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. If $\gamma = 1$ and $\beta = 0$ for all features, what are the mean and variance of $y$ along the batch dimension? Show this from the formulas.
2. If we set $\gamma_j = \sqrt{\sigma_{B,j}^2 + \epsilon}$ and $\beta_j = \mu_{B,j}$ for each feature $j$, what is $y_i$? What does this imply about batch norm’s representational capacity?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Let us now implement the forward pass.

## Part 2 (20 points, coding task)

**Do the following tasks.**

Implement `batchnorm_forward(x, gamma, beta, eps=1e-5)` that computes the batch normalization forward pass.

- `x`: tensor of shape `(B, D)`.
- `gamma`: tensor of shape `(D,)` — scale parameter.
- `beta`: tensor of shape `(D,)` — shift parameter.
- Returns `(y, cache)` where `y` has shape `(B, D)` and `cache` is a tuple of intermediate values needed for the backward pass.

The cache should include at least: `x, x_hat, mu, var, gamma, eps`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def batchnorm_forward(x, gamma, beta, eps=1e-5):
    """
    Batch normalization forward pass.
    x: (B, D), gamma: (D,), beta: (D,)
    Returns: (y, cache)
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
B, D = 32, 16
x = torch.randn(B, D)
gamma = torch.ones(D)
beta = torch.zeros(D)

y, cache = batchnorm_forward(x, gamma, beta)
assert y.shape == (B, D), f"Output shape: expected ({B}, {D}), got {y.shape}"
assert y.mean(dim=0).abs().max() < 1e-5, f"Mean should be ~0, got {y.mean(dim=0).abs().max():.6f}"
assert (y.var(dim=0, unbiased=False) - 1.0).abs().max() < 1e-5, "Variance should be ~1"

# Non-trivial gamma, beta
gamma2 = torch.full((D,), 2.0)
beta2 = torch.full((D,), 3.0)
y2, _ = batchnorm_forward(x, gamma2, beta2)
assert (y2.mean(dim=0) - 3.0).abs().max() < 1e-4, "Mean should be beta=3"
print("Part 2 passed!")

The backward pass through batch norm is more involved because $\mu_B$ and $\sigma_B^2$ depend on the entire batch, coupling every sample’s gradient to every other sample.

## Part 3 (25 points, coding task)

**Do the following tasks.**

Implement `batchnorm_backward(dy, cache)` that computes gradients for the backward pass.

- `dy`: tensor of shape `(B, D)` — gradient of loss w.r.t. output $y$.
- `cache`: tuple from the forward pass.
- Returns `(dx, dgamma, dbeta)` where:
  - `dx`: shape `(B, D)` — gradient w.r.t. input $x$.
  - `dgamma`: shape `(D,)` — gradient w.r.t. $\gamma$.
  - `dbeta`: shape `(D,)` — gradient w.r.t. $\beta$.

Key formulas:

$$\frac{\partial L}{\partial \gamma} = \sum_{i=1}^{B} \frac{\partial L}{\partial y_i} \odot \hat{x}_i$$

$$\frac{\partial L}{\partial \beta} = \sum_{i=1}^{B} \frac{\partial L}{\partial y_i}$$

$$\frac{\partial L}{\partial x_i} = \frac{\gamma}{\sqrt{\sigma_B^2 + \epsilon}} \left( \frac{\partial L}{\partial \hat{x}_i} - \frac{1}{B}\sum_j \frac{\partial L}{\partial \hat{x}_j} - \frac{\hat{x}_i}{B} \sum_j \frac{\partial L}{\partial \hat{x}_j} \cdot \hat{x}_j \right)$$

where $\frac{\partial L}{\partial \hat{x}_i} = \frac{\partial L}{\partial y_i} \cdot \gamma$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def batchnorm_backward(dy, cache):
    """
    Batch normalization backward pass.
    dy: (B, D), cache: tuple from forward
    Returns: (dx, dgamma, dbeta)
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
B, D = 16, 8
x = torch.randn(B, D)
gamma = torch.randn(D)
beta = torch.randn(D)

y, cache = batchnorm_forward(x, gamma, beta)
dy = torch.randn(B, D)
dx, dgamma, dbeta = batchnorm_backward(dy, cache)

# Verify against PyTorch
bn = nn.BatchNorm1d(D, affine=True, momentum=None)
bn.weight.data = gamma.clone()
bn.bias.data = beta.clone()
bn.train()
x_ag = x.clone().requires_grad_(True)
y_ag = bn(x_ag)
y_ag.backward(dy)

assert torch.allclose(dx, x_ag.grad, atol=1e-4), f"dx max error: {(dx - x_ag.grad).abs().max():.6f}"
assert torch.allclose(dgamma, bn.weight.grad, atol=1e-4), "dgamma mismatch"
assert torch.allclose(dbeta, bn.bias.grad, atol=1e-4), "dbeta mismatch"
print("Part 3 passed!")

In practice, batch norm behaves differently during training and inference. During training it uses batch statistics; during inference it uses **running** (exponential moving average) statistics accumulated over training.

## Part 4 (20 points, coding task)

**Do the following tasks.**

Implement a complete `MyBatchNorm1d` class as an `nn.Module` that tracks running statistics.

Requirements:
- Learnable parameters: `gamma` ($\gamma$) initialized to 1, `beta` ($\beta$) initialized to 0, both of shape `(D,)`.
- Buffers (non-learnable, tracked via `register_buffer`): `running_mean` initialized to 0, `running_var` initialized to 1, both of shape `(D,)`.
- **Training mode** (`self.training == True`): use batch statistics; update running stats via exponential moving average with momentum $m = 0.1$:
  - $\mu_{\text{running}} \leftarrow (1 - m)\, \mu_{\text{running}} + m\, \mu_B$
  - $\sigma^2_{\text{running}} \leftarrow (1 - m)\, \sigma^2_{\text{running}} + m\, \sigma_B^2$
- **Eval mode** (`self.training == False`): use running statistics (no batch stats, no update).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, D) -> (B, D)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
D = 8
my_bn = MyBatchNorm1d(D)

# Training mode
my_bn.train()
x = torch.randn(32, D)
y = my_bn(x)
assert y.shape == (32, D)
assert not torch.allclose(my_bn.running_mean, torch.zeros(D)), "Running mean should be updated"

# Several batches with shifted mean
for _ in range(100):
    my_bn(torch.randn(32, D) + 5.0)

# Eval mode with batch size 1
my_bn.eval()
x_single = torch.randn(1, D) + 5.0
y_eval = my_bn(x_single)
assert y_eval.shape == (1, D)
assert not torch.isnan(y_eval).any(), "Eval mode should work with batch size 1"
print("Part 4 passed!")

Let us now compare training convergence with and without batch normalization.

## Part 5 (15 points, coding task)

**Do the following tasks.**

Train two identical MLPs on a 2D classification dataset — one **with** batch norm and one **without** — and compare convergence.

Architecture: `Linear(2, 64) -> [BN] -> ReLU -> Linear(64, 64) -> [BN] -> ReLU -> Linear(64, 2)`.

1. Use the dataset defined below (circular boundary).
2. Train both models for 200 epochs with SGD at `lr=0.1` and `nn.CrossEntropyLoss`.
3. Store loss histories as `loss_without_bn` and `loss_with_bn` (Python lists of 200 floats).

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
torch.manual_seed(42)
N = 500
X_train = torch.randn(N, 2)
y_train = ((X_train[:, 0]**2 + X_train[:, 1]**2) < 1).long()

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

# Model WITHOUT batch norm
model_no_bn = ...

# Model WITH batch norm (use nn.BatchNorm1d here since MyBatchNorm1d may not support autograd)
model_with_bn = ...

loss_without_bn = []  # list of 200 floats
loss_with_bn = []     # list of 200 floats

# Training loop
...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert len(loss_without_bn) == 200
assert len(loss_with_bn) == 200
assert loss_without_bn[-1] < loss_without_bn[0], "Loss without BN should decrease"
assert loss_with_bn[-1] < loss_with_bn[0], "Loss with BN should decrease"
print(f"Loss at epoch 50 - No BN: {loss_without_bn[49]:.4f}, With BN: {loss_with_bn[49]:.4f}")
print(f"Final loss - No BN: {loss_without_bn[-1]:.4f}, With BN: {loss_with_bn[-1]:.4f}")
print("Part 5 passed!")

Let us close with conceptual questions about batch norm’s design.

## Part 6 (15 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. **Batch norm vs. Layer norm.** Batch norm computes $\mu$ and $\sigma^2$ **per feature across the batch**; layer norm computes them **per sample across features**. For an input of shape `(B, D)`, state the shapes of $\mu$ and $\sigma^2$ for each normalization method.

2. **Training/inference discrepancy.** During training, batch norm uses mini-batch statistics. During inference, it uses running statistics. If the mini-batch size is very small (e.g., $B = 2$), the batch statistics are noisy. Explain why this is a problem and how **Group Normalization** addresses it.

3. **Gradient coupling.** During the backward pass, the gradient $\frac{\partial L}{\partial x_i}$ depends on all other samples in the batch (through $\mu_B$ and $\sigma_B^2$). Is this coupling a feature or a bug? Give one argument in favour and one against.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """